# Week 2: Exercise 4 - Error Handling & Retries

**Goal:** Make your agent resilient to API failures.


In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai",
    api_key=os.environ.get("GEMINI_API_KEY"),
)
print("API client ready")


API client ready


## Step 1: Implement ErrorClassifier

**TODO:** Classify errors as retryable or permanent.
- Retryable: rate limit, timeout, 502, 503, 504, 429, overloaded
- Permanent: 401, 403, invalid api key, invalid model
- Return tuple: (error_type_string, is_retryable_bool)


In [2]:
class ErrorClassifier:
    """Classify errors as retryable or permanent."""

    @staticmethod
    def classify(error):
        """TODO: Return (error_type, is_retryable)."""
        # Hint: two keyword lists, checked case-insensitively (str(error).lower())
        #       retryable = ["rate limit", "timeout", "502", "503", "504", "429", "overloaded"]
        #       permanent = ["401", "403", "invalid api key", "invalid model"]
        #       return ("retryable", True) / ("permanent", False) / ("unknown", False)

        retryable = ["rate limit", "timeout", "502", "503", "504", "429", "overloaded"]
        for err_type in retryable:
            if err_type.lower() in str(error).lower():
                return ("retryable", True)
        permanent = ["401", "403", "invalid api key", "invalid model"]
        for err_type in permanent:
            if err_type.lower() in str(error).lower():
                return ("permanent", False)
            
        return ("unknown", False)
        


## Step 2: Understand Exponential Backoff

Formula: `delay = base_delay * (2 ** attempt)`


## Step 3: Implement safe_api_call

**TODO:** API call with retry logic and exponential backoff.

1. Loop max_retries + 1 times
2. Try the call — success: return response
3. On failure: classify the error
4. If not retryable or last attempt: return None
5. Otherwise: sleep, retry


In [3]:
import time

def safe_api_call(messages, max_retries=3, base_delay=1.0):
    """TODO: Make an API call with retry logic."""
    # Hint: for attempt in range(max_retries + 1):
    #           try: return client.chat.completions.create(...)
    #           except: classify; if not retryable or last attempt -> return None
    #                   else time.sleep(base_delay * 2**attempt)   <- exponential backoff
    for attempt in range(max_retries + 1):
        try:
            response = client.chat.completions.create(
                model = os.environ.get("GEMINI_3.6_MODEL"),
                messages = messages
            )

            return response
        except Exception as E:
            err_type, retryable = ErrorClassifier.classify(E)
            if not retryable or attempt == max_retries:
                print(f"Giving up on error {E}")
                return None

            delay = base_delay * (2 ** attempt)
            time.sleep(delay)


## Step 4: Test It


In [4]:
# Test 1: ErrorClassifier
t, r = ErrorClassifier.classify(Exception("429 Rate limit exceeded"))
assert r == True, "429 should be retryable"
print(f"  '429 rate limit' -> {t}, retryable={r}")

t, r = ErrorClassifier.classify(Exception("401 Unauthorized"))
assert r == False, "401 should be permanent"
print(f"  '401 unauthorized' -> {t}, retryable={r}")
print("Test 1 passed")


  '429 rate limit' -> retryable, retryable=True
  '401 unauthorized' -> permanent, retryable=False
Test 1 passed


In [5]:
# Test 2: Successful API call
messages = [{"role": "user", "content": "What is 2+2?"}]
response = safe_api_call(messages)
assert response is not None
print(f"  Response: {response.choices[0].message.content[:50]}")
print("Test 2 passed")


  Response: 2 + 2 = 4
Test 2 passed


In [6]:
# Test 3: Permanent error returns None immediately
print("--- Test 3: Permanent error (no retries) ---")
import time

def safe_api_call_bad_model(messages, max_retries=3, base_delay=0.1):
    for attempt in range(max_retries + 1):
        try:
            response = client.chat.completions.create(
                model="THIS_MODEL_DOES_NOT_EXIST",
                messages=messages
            )
            return response
        except Exception as e:
            error_type, is_retryable = ErrorClassifier.classify(e)
            print(f"  Attempt {attempt+1} failed: {error_type} - {str(e)[:60]}")
            if not is_retryable or attempt == max_retries:
                print(f"  Giving up ({error_type}).")
                return None
            delay = base_delay * (2 ** attempt)
            time.sleep(delay)

start = time.time()
result = safe_api_call_bad_model([{"role": "user", "content": "test"}])
elapsed = time.time() - start
assert result is None
assert elapsed < 2.0
print(f"  Took {elapsed:.2f}s (no retries)")
print("Test 3 passed")


--- Test 3: Permanent error (no retries) ---
  Attempt 1 failed: unknown - Error code: 400 - [{'error': {'code': 400, 'message': '* Gen
  Giving up (unknown).
  Took 0.17s (no retries)
Test 3 passed


## Key Takeaways
- Always classify errors before retrying
- Exponential backoff prevents overwhelming the server
